# Olist Raw Data — Cross-Table Analysis and Outlier Deep-Dives

Day 4 of Phase 1. Builds on per-table profiling in `01_data_profiling.ipynb`.

**Purpose:**
1. Investigate outliers and edge cases mentioned but not deeply explored on Day 3
2. Verify cross-table consistency at the row level (joins, multi-grain reconciliation)
3. Translate findings into final per-finding decisions (drop / keep / flag / transform) for the staging layer
4. Check business question feasibility against the data we have

**Builds on:** 18 data quality findings from Days 2–3 documented in `docs/data_quality.md`.

In [1]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

def find_project_root(start: Path, marker: str = ".env") -> Path:
    for parent in [start, *start.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} walking up from {start}")

PROJECT_ROOT = find_project_root(Path.cwd())
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 100)

print(f"Project root: {PROJECT_ROOT}")
print("Setup complete.")

Project root: /Users/gowthamir/Projects/ecommerce-analytics-pipeline
Setup complete.


---

## Outlier Deep-Dive 1: Delivery Time

Quantify the customer-facing delivery experience. Two metrics:
- **Actual delivery days** (purchase → customer delivery)
- **Delivery vs. estimate** (negative = early, positive = late)

Focus areas:
- Distribution shape and percentiles
- Extreme outliers (>60 days)
- How often is Olist late vs. early?

In [2]:
orders = pd.read_sql("""
    SELECT order_id, order_status,
           order_purchase_timestamp,
           order_delivered_customer_date,
           order_estimated_delivery_date
    FROM raw.orders
    WHERE order_delivered_customer_date IS NOT NULL
""", engine)

print(f"Delivered orders (with delivery timestamp): {len(orders):,}")

# Delivery time in days
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Delivery vs estimate (positive = late)
orders["vs_estimate_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

print("\nActual delivery time (days):")
print(orders["delivery_days"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(2))

print("\nDelivery vs. estimate (days; positive = late, negative = early):")
print(orders["vs_estimate_days"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(2))

print(f"\nOrders delivered EARLIER than estimate: {(orders['vs_estimate_days'] < 0).sum():,} ({(orders['vs_estimate_days'] < 0).mean()*100:.1f}%)")
print(f"Orders delivered ON or AFTER estimate: {(orders['vs_estimate_days'] >= 0).sum():,}")
print(f"Orders delivered >7 days LATE: {(orders['vs_estimate_days'] > 7).sum():,}")
print(f"Orders delivered >30 days LATE: {(orders['vs_estimate_days'] > 30).sum():,}")

Delivered orders (with delivery timestamp): 96,476

Actual delivery time (days):
count    96476.00
mean        12.56
std          9.55
min          0.53
50%         10.22
75%         15.72
90%         23.10
95%         29.28
99%         46.05
max        209.63
Name: delivery_days, dtype: float64

Delivery vs. estimate (days; positive = late, negative = early):
count    96476.00
mean       -11.18
std         10.19
min       -146.02
5%         -25.94
25%        -16.24
50%        -11.95
75%         -6.39
95%          3.82
max        188.98
Name: vs_estimate_days, dtype: float64

Orders delivered EARLIER than estimate: 88,649 (91.9%)
Orders delivered ON or AFTER estimate: 7,827
Orders delivered >7 days LATE: 3,346
Orders delivered >30 days LATE: 360


In [4]:
very_late = orders[orders["vs_estimate_days"] > 30].copy()
print(f"Orders >30 days late: {len(very_late)}")

# Join with reviews to see score distribution
reviews = pd.read_sql("SELECT order_id, review_score FROM raw.order_reviews", engine)
late_with_reviews = very_late.merge(reviews, on="order_id", how="left")

# Some late orders may not have reviews
has_review = late_with_reviews[late_with_reviews["review_score"].notnull()]
print(f" which of them have at least one review: {len(has_review)}")
print()
print("Review score distribution for >30-day-late orders:")
print(has_review["review_score"].value_counts().sort_index())
print()
print(f"Mean review score (>30 days late):     {has_review['review_score'].mean():.2f}")

# Compare to overall mean
all_with_reviews = orders.merge(reviews, on="order_id", how="left")
all_with_reviews = all_with_reviews[all_with_reviews["review_score"].notnull()]
print(f"Mean review score (all delivered):     {all_with_reviews['review_score'].mean():.2f}")
print(f"Mean review score (on-time/early):     {all_with_reviews[all_with_reviews['vs_estimate_days'] <= 0]['review_score'].mean():.2f}")

Orders >30 days late: 360
 which of them have at least one review: 345

Review score distribution for >30-day-late orders:
review_score
1.0    221
2.0     16
3.0     32
4.0     31
5.0     45
Name: count, dtype: int64

Mean review score (>30 days late):     2.02
Mean review score (all delivered):     4.16
Mean review score (on-time/early):     4.29


### Key insight (business, not DQ)

- 360 orders delivered >30 days after estimate
- These orders have mean review score 2.02 vs 4.29 for on-time deliveries
- 64% of late-delivery reviews are 1-star
- 45 late deliveries still received 5 stars — so delay is *strongly* but not *exclusively* the driver

This is a high-leverage analytical relationship the star schema must make trivially joinable: `fact_orders.delivery_lateness_days` and `fact_reviews.review_score`.

In [5]:
extreme_late = orders[orders["delivery_days"] > 100].sort_values("delivery_days", ascending=False)
print(f"Orders with delivery_days > 100: {len(extreme_late)}")
print()
print(extreme_late.head(10)[
    ["order_id", "order_status", "order_purchase_timestamp", 
     "order_delivered_customer_date", "delivery_days", "vs_estimate_days"]
])

Orders with delivery_days > 100: 64

                               order_id order_status order_purchase_timestamp order_delivered_customer_date  delivery_days  vs_estimate_days
19031  ca07593549f1816d26a572e06dc1eab6    delivered      2017-02-21 23:31:27           2017-09-19 14:36:39     209.628611        181.608785
53965  1b3190b2dfa9d789e1f14c05b647a14a    delivered      2018-02-23 14:57:35           2018-09-19 23:24:07     208.351759        188.975081
59782  440d0d17af552815d15a9e41abe49359    delivered      2017-03-07 23:59:51           2017-09-19 15:12:50     195.634016        165.633912
68211  2fb597c2f772eca01b1f5c561bf6cc7b    delivered      2017-03-08 18:09:02           2017-09-19 14:33:17     194.850174        155.606447
86463  285ab9426d6982034523a855f55a885e    delivered      2017-03-08 22:47:40           2017-09-19 14:00:04     194.633611        166.583380
37373  0f4519c5f1c541ddec9f21b3bddd533a    delivered      2017-03-09 13:26:57           2017-09-19 14:38:21     194.0

### Delivery time summary

- Median delivery: 10 days. 95% within 29 days. 99% within 46 days.
- 91.9% delivered earlier than estimate (sandbagged estimates)
- 8.1% late; 3.5% >7 days late; 0.4% >30 days late
- Late delivery is a major review-score driver: >30-day-late orders have mean score 2.02 vs 4.29 on-time
- 64 extreme-tail "deliveries" (>100 days) are administrative batch closures, not actual deliveries (DQ-019)

Key downstream consequence: `fact_orders` should include a derived `delivery_lateness_days` and a `is_batch_resolved` flag.

---

## Outlier Deep-Dive 2: Price Extremes

Day 3 inspected the top 10 priciest items (all plausible). This dive looks at:
- The bottom of the price distribution (cheapest items — sanity check)
- Whether top-priced items cluster in any one seller or category
- Whether price × freight ratios reveal pricing anomalies

In [6]:
items = pd.read_sql("SELECT * FROM raw.order_items", engine)
print(f"Total items: {len(items):,}")
print()

# Bottom of distribution
print("Cheapest 10 items:")
print(items.nsmallest(10, "price")[
    ["order_id", "product_id", "seller_id", "price", "freight_value"]
])
print()
print(f"Items with price < $5: {(items['price'] < 5).sum():,}")
print(f"Items with price < $1: {(items['price'] < 1).sum():,}")
print()

# Freight-to-price ratio — find cases where freight is huge relative to price
items["freight_to_price_ratio"] = items["freight_value"] / items["price"]
print("Freight-to-price ratio distribution:")
print(items["freight_to_price_ratio"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(3))
print()
print(f"Items where freight > price: {(items['freight_value'] > items['price']).sum():,}")
print(f"Items where freight is >= 2x the price: {(items['freight_to_price_ratio'] >= 2).sum():,}")
print(f"Items where freight is >= 5x the price: {(items['freight_to_price_ratio'] >= 5).sum():,}")

Total items: 112,650

Cheapest 10 items:
                               order_id                        product_id                         seller_id  price  freight_value
27652  3ee6513ae7ea23bdfab5b9ab60bffcb5  8a3254bee785a526d548a81a9bc3c9be  96804ea39d96eb908e7c3afdb671bb9e   0.85          18.23
48625  6e864b3f0ec71031117ad4cf46b7f2a1  8a3254bee785a526d548a81a9bc3c9be  96804ea39d96eb908e7c3afdb671bb9e   0.85          18.23
87081  c5bdd8ef3c0ec420232e668302179113  8a3254bee785a526d548a81a9bc3c9be  96804ea39d96eb908e7c3afdb671bb9e   0.85          22.30
57297  8272b63d03f5f79c56e9e4120aec44ef  270516a3f41dc035aa87d220228f844c  2709af9587499e95e803a6498a5a56e9   1.20           7.89
57298  8272b63d03f5f79c56e9e4120aec44ef  05b515fdc76e888aada3c6d66c201dff  2709af9587499e95e803a6498a5a56e9   1.20           7.89
57299  8272b63d03f5f79c56e9e4120aec44ef  05b515fdc76e888aada3c6d66c201dff  2709af9587499e95e803a6498a5a56e9   1.20           7.89
57300  8272b63d03f5f79c56e9e4120aec44ef  05b515fd

In [7]:
extreme_freight = items[items["freight_to_price_ratio"] >= 5].sort_values("freight_to_price_ratio", ascending=False)
print(f"Items with freight ≥ 5x price: {len(extreme_freight)}")
print()
print("Top 10 most extreme freight-to-price ratios:")
print(extreme_freight.head(10)[
    ["order_id", "product_id", "seller_id", "price", "freight_value", "freight_to_price_ratio"]
].round(2))
print()

# Quick check: are these cheap items shipped to faraway states? 
# Join with orders → customers to see destination state
extreme_with_dest = pd.read_sql("""
    SELECT i.order_id, i.product_id, i.price, i.freight_value,
           (i.freight_value / i.price) AS freight_ratio,
           c.customer_state, s.seller_state
    FROM raw.order_items i
    JOIN raw.orders o ON i.order_id = o.order_id
    JOIN raw.customers c ON o.customer_id = c.customer_id
    JOIN raw.sellers s ON i.seller_id = s.seller_id
    WHERE i.freight_value >= 5 * i.price
    ORDER BY freight_ratio DESC
    LIMIT 10
""", engine)
print("Same top 10 with origin/destination context:")
print(extreme_with_dest.round(2))

Items with freight ≥ 5x price: 41

Top 10 most extreme freight-to-price ratios:
                                order_id                        product_id                         seller_id  price  freight_value  freight_to_price_ratio
87081   c5bdd8ef3c0ec420232e668302179113  8a3254bee785a526d548a81a9bc3c9be  96804ea39d96eb908e7c3afdb671bb9e   0.85          22.30                   26.24
48625   6e864b3f0ec71031117ad4cf46b7f2a1  8a3254bee785a526d548a81a9bc3c9be  96804ea39d96eb908e7c3afdb671bb9e   0.85          18.23                   21.45
27652   3ee6513ae7ea23bdfab5b9ab60bffcb5  8a3254bee785a526d548a81a9bc3c9be  96804ea39d96eb908e7c3afdb671bb9e   0.85          18.23                   21.45
110535  fb265b2dc558a56445dfc48f8224e201  baf25ed4f8f70238cc87230379471454  128f9bfbe4c7d5185033914b1de3d39a   9.90         121.22                   12.24
94495   d642656598ae928a250620315d19e87e  b07fffe072c9adc235a35d8da7c0584d  dd533b429f380718b70ad9922c294bae   4.99          37.04               

### Price extremes summary

- Cheapest items ($0.85–$1.20) are legitimate micro-priced products from specialized sellers
- 117 items priced under $5; 3 priced under $1
- Freight-to-price ratio: median 0.23 (freight typically 23% of item price)
- 4,124 items (3.7%) have freight > price
- 41 items have freight ≥ 5× price — all explainable by either:
  - Minimum freight charges on micro-priced items
  - Long-distance shipping to remote/southern states (Pará, Rio Grande do Sul)

No data quality findings. Confirms expected freight pricing dynamics. Useful business context: **micro-priced items are economically unviable to ship long distances** — a useful seller-strategy insight for the README.

---

## Outlier Deep-Dive 3: Geolocation Aggregation Strategy

DQ-017 (city name inconsistency) and DQ-016 (out-of-bounds coords) both make this a critical decision: when collapsing 1M geolocation rows into one row per zip prefix for `dim_geolocation`, how do we aggregate?

**Compare median vs. mean** for an example zip and look at the impact of outliers.

In [9]:
geo = pd.read_sql("SELECT * FROM raw.geolocation", engine)

# Compare aggregation strategies for a few representative zip prefixes
sample_zips = ["01001", "01037", "06900", "28155"]
# 06900 had 5 city variants (DQ-017)
# 28155 had an out-of-Brazil outlier (DQ-016)

for zip_pref in sample_zips:
    subset = geo[geo["geolocation_zip_code_prefix"] == zip_pref]
    if len(subset) == 0:
        continue
    print(f" Zip prefix {zip_pref} ({len(subset)} rows)")
    print(f"  Lat — mean: {subset['geolocation_lat'].mean():.4f}, "
          f"median: {subset['geolocation_lat'].median():.4f}, "
          f"std: {subset['geolocation_lat'].std():.4f}")
    print(f"  Lng — mean: {subset['geolocation_lng'].mean():.4f}, "
          f"median: {subset['geolocation_lng'].median():.4f}, "
          f"std: {subset['geolocation_lng'].std():.4f}")
    print()

# Quantify: how often do mean and median diverge significantly?
agg = geo.groupby("geolocation_zip_code_prefix").agg(
    lat_mean=("geolocation_lat", "mean"),
    lat_median=("geolocation_lat", "median"),
    lng_mean=("geolocation_lng", "mean"),
    lng_median=("geolocation_lng", "median"),
    row_count=("geolocation_lat", "count"),
)
agg["lat_diff_km"] = (agg["lat_mean"] - agg["lat_median"]).abs() * 111  # 1 deg lat ≈ 111 km
agg["lng_diff_km"] = (agg["lng_mean"] - agg["lng_median"]).abs() * 111  # rough

print("Mean-vs-median divergence (km equivalent):")
print(f"  Lat diff > 1 km: {(agg['lat_diff_km'] > 1).sum():,} zip prefixes ({(agg['lat_diff_km'] > 1).mean()*100:.1f}%)")
print(f"  Lat diff > 10 km: {(agg['lat_diff_km'] > 10).sum():,} zip prefixes")
print(f"  Lat diff > 100 km: {(agg['lat_diff_km'] > 100).sum():,} zip prefixes")

 Zip prefix 01001 (26 rows)
  Lat — mean: -23.5502, median: -23.5504, std: 0.0007
  Lng — mean: -46.6340, median: -46.6340, std: 0.0003

 Zip prefix 01037 (26 rows)
  Lat — mean: -23.5454, median: -23.5454, std: 0.0010
  Lng — mean: -46.6389, median: -46.6388, std: 0.0011

 Zip prefix 06900 (447 rows)
  Lat — mean: -23.8351, median: -23.8339, std: 0.0357
  Lng — mean: -46.8141, median: -46.8152, std: 0.0166

 Zip prefix 28155 (5 rows)
  Lat — mean: -11.3147, median: -21.7559, std: 30.5975
  Lng — mean: -34.7251, median: -43.3791, std: 28.0074

Mean-vs-median divergence (km equivalent):
  Lat diff > 1 km: 1,178 zip prefixes (6.2%)
  Lat diff > 10 km: 167 zip prefixes
  Lat diff > 100 km: 52 zip prefixes


### Geolocation aggregation decision

For `dim_geolocation`, **use median** of lat/lng per zip prefix (not mean).

**Evidence:**
- For normal zip prefixes (>99% of them), mean and median agree to within meters — choice is inconsequential
- For zip prefixes containing DQ-016 out-of-bounds coordinates, mean is pulled hundreds of km off-position
- 52 zip prefixes would have dim_geolocation coordinates >100 km wrong if mean were used
- Median is robust to the 42 known outliers without requiring upstream filtering

This decision is recorded in the DQ-016 implication section and will be formalized in the Day 6 ADR.